# PhishSentry — email model retrain **v4**

Fixes two weaknesses that the expanded out-of-distribution evaluation exposed in
v3 (n=150 per category, Wilson intervals):

| Category | v3 | target |
|---|---:|---:|
| Phishing — overt | 80.0% [73, 86] | ≥ 90% |
| **Phishing — subtle / BEC** | **15.6% [9, 24]** | **≥ 70%** |
| Legitimate — transactional | 100% | ≥ 97% |
| **Legitimate — security notices** | **90.0% [84, 94]** | **≥ 97%** |
| Legitimate — work / personal | 100% | ≥ 97% |
| Legitimate — marketing | 98.9% | ≥ 95% |

**Method note.** BEC augmentation is generated, because real BEC corpora are
scarce. Training uses **pool A** templates; the gate uses **pool B** templates
that differ in structure, not just in filled values. Pool B is never trained on.
Passing the gate means the model generalises across BEC *phrasing structures*,
which is weaker than "detects real BEC" — record that distinction in
EVALUATION.md.

Runtime → Change runtime type → **T4 GPU**.

In [ ]:
!pip -q install transformers==4.44.2 datasets scikit-learn pandas beautifulsoup4 kaggle
import torch, numpy as np, pandas as pd, re, html, glob, os, shutil, random, itertools
from bs4 import BeautifulSoup
random.seed(42); np.random.seed(42); torch.manual_seed(42)
print('CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable GPU: Runtime -> Change runtime type -> T4 GPU'

## 1. Uploads + base dataset
Upload (1) `kaggle.json` and (2) `emails_labeled_updated.csv` (your hand-labelled inbox export).

In [ ]:
from google.colab import files
print('>>> Upload kaggle.json:'); files.upload()
print('>>> Upload emails_labeled_updated.csv:'); files.upload()
kj=[f for f in os.listdir('.') if f.lower().startswith('kaggle') and f.endswith('.json')][0]
os.makedirs('/root/.kaggle',exist_ok=True); shutil.copy(kj,'/root/.kaggle/kaggle.json'); os.chmod('/root/.kaggle/kaggle.json',0o600)
!kaggle datasets download -d naserabdullahalam/phishing-email-dataset --unzip -p ./data
for f in glob.glob('./data/**/*.csv',recursive=True):
    try: print(f,'->',list(pd.read_csv(f,nrows=2).columns))
    except Exception as e: print(f,'ERR',e)

## 2. Augmentation generators — **POOL A (training only)**

Pool A structures are deliberately different from the pool B structures used by
the gate. If you edit these, keep them disjoint or the gate stops measuring
generalisation.

In [ ]:
BRANDS=["PayPal","Amazon","Netflix","Apple","Microsoft","Google","DHL","FedEx",
        "HDFC Bank","ICICI Bank","Instagram","LinkedIn","Dropbox","Spotify",
        "Coinbase","Ledger","Steam","Zoom","Outlook","Adobe"]
NAMES=["Abhishek","Priya","Sam","Alex","Ravi","Meera","Daniel","Aisha","Tom","Nisha"]
CITIES=["Chennai","Delhi","Mumbai","London","Singapore","Toronto","Berlin","Dubai"]
ROLES=["Finance","Accounts Payable","Procurement","HR","IT Support","Payroll"]

def _f(t,**kw):
    out=t
    for k,v in kw.items(): out=out.replace("{"+k+"}",str(v))
    while "{num}" in out: out=out.replace("{num}",str(random.randint(1000,99999)),1)
    while "{amount}" in out:
        out=out.replace("{amount}",random.choice(["$4,800","$12,500","₹2,40,000",
                        "£7,300","$860.00","₹95,000"]),1)
    while "{day}" in out:
        out=out.replace("{day}",random.choice(["Monday","Wednesday","Friday",
                        "14 August","2 September","tomorrow"]),1)
    while "{time}" in out:
        out=out.replace("{time}",random.choice(["11:00","4pm","17:30","09:15"]),1)
    return out

# ---------------------------------------------------------------- POOL A: BEC
# Structures: executive-impersonation payment requests, vendor bank-detail
# changes, procurement/PO framing, payroll redirection, gift-card pretexts,
# confidentiality framing, thread-hijack style short replies.
POOL_A_BEC=[
 "Are you at your desk? I need a payment released today and cannot take calls — reply here and I'll send the beneficiary details.",
 "Confidential: we're finalising an acquisition. Do not discuss with anyone. I need {amount} moved to a holding account before {day}.",
 "Following the audit, {role} has updated our remittance account. Please use the details below for all future invoices, effective immediately.",
 "Our bank has flagged the old account. New beneficiary details attached — please update the vendor master before the next payment run.",
 "Can you tell me our approved supplier payment cut-off? I have an urgent one to push through and don't want it missed.",
 "{name}, please arrange a same-day transfer of {amount}. It relates to a deposit that must clear before the contract lapses.",
 "Reminder that PO #{num} is overdue. Kindly remit to the account on the revised invoice attached and confirm once sent.",
 "Payroll change request: please redirect my salary to the account below from this month onward. New bank, same name.",
 "I'm boarding shortly. Could you purchase {amount} in gift cards for the client gifts and send me the codes? I'll reimburse on return.",
 "Short one — is the wire I asked about yesterday done? Client is chasing and I'd rather not delay this further.",
 "The supplier says invoice #{num} is unpaid. Their new banking details are in the attachment; please settle today and confirm.",
 "As discussed with {role}, treat this as approved. Transfer {amount} to the beneficiary in the attached form and send me the reference.",
 "Urgent from the CFO's office: hold all payments to the previous account. Use the updated details circulated below.",
 "Please review the attached remittance and confirm the account change has been applied in the ledger before {day}.",
 "Our finance system is mid-migration, so process this one manually: {amount} to the account in the attachment, reference {num}.",
 "Quick favour — can you confirm which of our vendors are set up for international transfer? Need it for a payment going out {day}.",
 "The invoice attached was rejected by our bank. Resubmit payment to the corrected IBAN provided and let me know when cleared.",
 "{name}, the auditors need the payment confirmation for {amount} by {time}. Process and forward the SWIFT copy.",
 "Notice from {role}: banking details for {brand} have changed. All outstanding invoices should be paid to the new account below.",
 "Are you able to action a transfer outside the normal approval chain? It's time-sensitive and I'll sign off retrospectively.",
 "Attached: revised vendor onboarding form with updated account information. Please process and destroy earlier copies.",
 "Following up on my last message about the {amount} payment. Has it gone out? Please prioritise it over the other items in the queue.",
 "Contract signature needed — open the shared document and authenticate with your work credentials to view and sign.",
 "{role} has requested that all staff reconfirm their bank details for the new payroll platform using the form below.",
 "This is a private request and should not be copied to the wider team. I need {amount} released against the attached schedule.",
]

# ------------------------------------------- POOL A: legitimate security notices
# Structures deliberately different from pool B: app-approval prompts, OTP
# phrasing variants, recovery-info changes, session lists, admin notices.
POOL_A_SEC=[
 "A sign-in attempt from {city} is waiting for approval. Open the app and tap Approve if it was you, or Deny to block it.",
 "Your one-time passcode is {num}. It is valid for ten minutes. We will never ask you to share it.",
 "Your recovery phone number was updated. If you did not make this change, review your security settings.",
 "You have three active sessions. Sign out of any device you no longer use from the Security page.",
 "Verification complete — your email address has been confirmed. Nothing further is required.",
 "Your account recovery codes were regenerated on {day}. Store them somewhere safe.",
 "Sign-in from a new browser on {day} at {time}. This message is for your records.",
 "Your authenticator app was linked successfully. Two-step verification is now active on your account.",
 "Card ending {num} was used for a purchase of {amount} on {day}. This notification is informational.",
 "A standing instruction on your account is due on {day}. No action is needed if the details are unchanged.",
 "Statement for the period ending {day} is ready. Download it from the Documents section when convenient.",
 "Your registered email for security alerts has changed. Contact support if this was unexpected.",
 "Scheduled maintenance on {day} between {time} and midnight. Sign-in may be briefly unavailable.",
 "We have retired support for older TLS versions. Modern browsers are unaffected.",
 "Your subscription payment for {amount} was successful. The next renewal is on {day}.",
 "Password change confirmed on {day}. If you did not do this, use the recovery link on the sign-in page.",
 "New device registered: a tablet in {city}. Manage trusted devices in your account settings.",
 "Your data export is ready and will remain available for seven days.",
 "Two-step verification was turned on for your account. You will be asked for a code at each new sign-in.",
 "Delivery of your card is scheduled for {day}. Activate it in the app once it arrives.",
]

def gen_bec(n):
    out=set()
    openers=["", "Hi {name},", "{name},", "Hello,", "Morning,", "Hi,"]
    subjects=["Payment request","Urgent — action needed","Invoice update",
              "Bank details change","Confidential","Re: outstanding payment",
              "Approval required","Vendor account update","Quick request"]
    combos=list(itertools.product(openers,POOL_A_BEC,subjects))
    random.shuffle(combos)
    for op,body,subj in combos:
        if len(out)>=n: break
        kw=dict(name=random.choice(NAMES),role=random.choice(ROLES),
                brand=random.choice(BRANDS),city=random.choice(CITIES))
        text=f"Subject: {subj}\n\n{_f(op,**kw)} {_f(body,**kw)}".strip()
        out.add(text)
    return list(out)

def gen_sec_legit(n):
    out=set()
    subjects=["Security notification","Sign-in alert","Verification code",
              "Account update","Your account","Notice","Confirmation"]
    combos=list(itertools.product(POOL_A_SEC,subjects))
    random.shuffle(combos)
    while len(out)<n:
        before=len(out)
        for body,subj in combos:
            if len(out)>=n: break
            kw=dict(name=random.choice(NAMES),role=random.choice(ROLES),
                    brand=random.choice(BRANDS),city=random.choice(CITIES))
            b=random.choice(BRANDS)
            out.add(f"Subject: {b} {subj}\n\n{_f(body,**kw)}")
        if len(out)==before: break
    return list(out)

BEC_AUG=gen_bec(4000)
SEC_AUG=gen_sec_legit(4000)
print('pool A BEC augmentation      :',len(BEC_AUG))
print('pool A security-legit augment:',len(SEC_AUG))
print('\nsample BEC:\n',BEC_AUG[0][:220])
print('\nsample security-legit:\n',SEC_AUG[0][:220])

## 3. Build the training set

Same construction as v3, plus the two augmentation blocks. Transactional
oversampling is retained — it is what fixed the v2 regression and must not be
dropped.

In [ ]:
frames=[]
def comb(df):
    s=df['subject'].fillna('') if 'subject' in df.columns else ''
    b=df['body'].fillna('') if 'body' in df.columns else ''
    return (s+' '+b).str.strip()

for f in glob.glob('./data/**/*.csv',recursive=True):
    d=pd.read_csv(f); cols=[c.lower() for c in d.columns]
    if 'label' not in cols: continue
    if 'text_combined' in cols: text=d['text_combined'].astype(str)
    elif 'body' in cols: text=comb(d)
    elif 'text' in cols: text=d['text'].astype(str)
    else: continue
    frames.append(pd.DataFrame({'text':text,'label':d['label'].astype(int)}))
    print(f,'ok',len(d))

# hand-labelled modern inbox mail (high value -> oversampled)
m=pd.read_csv('emails_labeled_updated.csv')
modern_legit=m[m['verified_label']=='legit'][['text']].copy(); modern_legit['label']=0
modern_spam =m[m['verified_label']=='spam' ][['text']].copy(); modern_spam['label'] =1
print('modern legit:',len(modern_legit),' modern spam:',len(modern_spam))
OVERSAMPLE=15
frames += [modern_legit]*OVERSAMPLE + [modern_spam]*OVERSAMPLE

# curated legitimate transactional templates (fixed the v2 regression)
TXN=[ "Your order #{n} has shipped and should arrive {d}.",
      "Thanks for your purchase. Your receipt is attached.",
      "Your package was delivered {d}.",
      "Your monthly statement is now available.",
      "Your subscription renews on the {n}th. No action needed.",
      "Your refund has been processed and will appear in 5-7 days.",
      "Your booking is confirmed. Reference {n}.",
      "Your invoice has been paid. Thank you.",
      "Your flight is on schedule. Boarding at {d}.",
      "Your table is confirmed for {d}." ]
txn_rows=[]
for t in TXN:
    for _ in range(60):
        b=random.choice(BRANDS)
        txn_rows.append({'text':f"Subject: {b} order update\n\n"+
            t.replace("{n}",str(random.randint(1000,99999)))
             .replace("{d}",random.choice(["Thursday","Monday","14 August","6pm"])),
            'label':0})
frames.append(pd.DataFrame(txn_rows))
print('transactional templates:',len(txn_rows))

# NEW in v4 -- the two augmentation blocks
frames.append(pd.DataFrame({'text':BEC_AUG,'label':[1]*len(BEC_AUG)}))
frames.append(pd.DataFrame({'text':SEC_AUG,'label':[0]*len(SEC_AUG)}))

df=pd.concat(frames,ignore_index=True).dropna(subset=['text'])
df['text']=df['text'].astype(str)
df=df[df['text'].str.len()>20]
df=df.drop_duplicates(subset=['text'])
print('\nTOTAL:',len(df))
print(df.label.value_counts())

## 4. Normalise, tokenise, split

Normalisation must match `inference_service.py` exactly — strip `Subject:` and
HTML, replace URLs with `httpaddr`. A mismatch here is train/serve skew.

In [ ]:
def preprocess(t):
    t=html.unescape(str(t))
    if '<' in t and '>' in t: t=re.sub(r'<[^>]+>',' ',t)
    t=re.sub(r'^\s*subject\s*:\s*','',t,flags=re.I)
    t=re.sub(r'https?://\S+|www\.\S+',' httpaddr ',t)
    return re.sub(r'\s+',' ',t).strip()[:5000]

df['text']=df['text'].map(preprocess)
df=df[df['text'].str.len()>20].drop_duplicates(subset=['text']).sample(frac=1,random_state=42)
print(len(df)); print(df.label.value_counts())

from sklearn.model_selection import train_test_split
tr,te=train_test_split(df,test_size=0.2,random_state=42,stratify=df.label)
print('train',len(tr),'test',len(te))

from transformers import RobertaTokenizer
tok=RobertaTokenizer.from_pretrained('roberta-base')
import datasets
def enc(b): return tok(b['text'],truncation=True,padding='max_length',max_length=256)
dtr=datasets.Dataset.from_pandas(tr[['text','label']],preserve_index=False).map(enc,batched=True)
dte=datasets.Dataset.from_pandas(te[['text','label']],preserve_index=False).map(enc,batched=True)
dtr.set_format('torch',columns=['input_ids','attention_mask','label'])
dte.set_format('torch',columns=['input_ids','attention_mask','label'])

## 5. Train (class-weighted, as v3)

In [ ]:
from transformers import Trainer, TrainingArguments, RobertaForSequenceClassification
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score,precision_score,recall_score,accuracy_score

cw=compute_class_weight('balanced',classes=np.array([0,1]),y=tr.label.values)
W=torch.tensor(cw,dtype=torch.float).cuda(); print('weights[legit,phish]:',cw)
model=RobertaForSequenceClassification.from_pretrained('roberta-base',num_labels=2).cuda()

class WT(Trainer):
    def compute_loss(s,model,inputs,return_outputs=False,**k):
        lb=inputs.pop('labels') if 'labels' in inputs else inputs.pop('label')
        o=model(**inputs)
        loss=torch.nn.functional.cross_entropy(o.logits,lb,weight=W)
        return (loss,o) if return_outputs else loss

def mets(p):
    pr=p.predictions.argmax(-1)
    return {'acc':accuracy_score(p.label_ids,pr),'f1':f1_score(p.label_ids,pr),
            'precision':precision_score(p.label_ids,pr),'recall':recall_score(p.label_ids,pr)}

args=TrainingArguments(output_dir='./out',num_train_epochs=2,
    per_device_train_batch_size=32,per_device_eval_batch_size=64,
    learning_rate=2e-5,warmup_ratio=0.06,weight_decay=0.01,
    eval_strategy='epoch',save_strategy='no',logging_steps=200,
    fp16=True,report_to=[])
trainer=WT(model=model,args=args,train_dataset=dtr,eval_dataset=dte,compute_metrics=mets)
trainer.train()
print(trainer.evaluate())

## 6. **GATE — pool B templates, never trained on**

These structures do not appear in pool A. Every category must clear its bar or
the weights are not saved. The bars for the two new categories are the point of
this retrain; the others guard against regression.

In [ ]:
import math
def wilson(k,n,z=1.96):
    if n==0: return (0.0,0.0)
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n)
    m=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return ((c-m)/d,(c+m)/d)

# ---- POOL B: structures deliberately absent from pool A ----
B_SUBTLE=[
 "Hi {name}, following up on our conversation. I've shared the updated document via the portal: httpaddr",
 "Please find attached invoice #{num} for services rendered. Kindly process payment to the updated account details within 3 business days.",
 "{name}, I need you to process an urgent wire transfer before end of day. I'm in meetings, so email only. Details to follow.",
 "Your {brand} mailbox is almost full and you may stop receiving emails. Revalidate your credentials using the link below to continue.",
 "HR notice: please review and acknowledge the updated payroll document at httpaddr before {day}.",
]
B_OVERT=[
 "Dear Customer, we are writing to inform you that your account has been suspended. Immediate action is required. Click here to verify your identity: httpaddr",
 "Dear User, suspicious activity was found on your account. You must respond within 24 hours. Confirm your details now: httpaddr",
 "Attention: your payment method was declined. Act now to avoid service interruption. Update your payment information: httpaddr",
 "Congratulations! You have been selected for a refund of {amount}. Reply with your bank account number to claim.",
 "Your parcel is being held at customs. Pay the outstanding fee to release it: httpaddr",
]
B_SEC=[
 "We noticed a new sign-in to your account from a Windows device in Chennai. If this was you, no action is needed.",
 "You recently requested a password reset. The link below expires in 30 minutes. If you did not request this, you can safely ignore this email.",
 "A new device was added to your account on Monday. If you don't recognise it, review your security settings.",
 "Your two-factor authentication code is {num}. It expires in 10 minutes.",
 "A transaction of {amount} was made on your card ending 4821 at Chennai on Friday.",
 "Your password was changed successfully on Tuesday.",
]
B_TXN=[
 "Your order #{num} has shipped and should arrive Thursday.",
 "Thanks for your purchase. Your receipt for {amount} is attached.",
 "Your Amazon package was delivered. Rate your experience.",
 "Your monthly statement is now available in your account.",
 "Your subscription renews on the 15th. No action is needed.",
 "Your refund of {amount} has been processed and will appear in 5-7 days.",
]
B_WORK=[
 "Hi team, attaching the notes from today's standup. Let me know if I missed anything.",
 "Reminder: our 1:1 is scheduled for 3pm tomorrow.",
 "The quarterly report is ready for review.",
 "Meeting rescheduled to Friday 2pm. Calendar updated.",
]
B_MKT=[
 "This week at {brand}: three new features, and a look at what's next.",
 "Our summer sale starts Monday. Up to 40% off selected items.",
 "You're invited to our webinar on Friday at 2pm. Registration is free.",
]

def expand(pool,n):
    out=[]
    for i in range(n):
        t=pool[i%len(pool)]
        out.append(_f(t,name=random.choice(NAMES),brand=random.choice(BRANDS),
                      role=random.choice(ROLES),city=random.choice(CITIES)))
    return out

def score(texts):
    model.eval(); ps=[]
    for i in range(0,len(texts),64):
        e=tok([preprocess(x) for x in texts[i:i+64]],truncation=True,
              padding=True,max_length=256,return_tensors='pt').to('cuda')
        with torch.no_grad():
            ps+=torch.softmax(model(**e).logits,dim=1)[:,1].cpu().tolist()
    return ps

CHECKS=[('Phishing - overt',      expand(B_OVERT,150), True,  0.90),
        ('Phishing - subtle/BEC', expand(B_SUBTLE,150),True,  0.70),
        ('Legit - transactional',  expand(B_TXN,150),   False, 0.97),
        ('Legit - security notes', expand(B_SEC,150),   False, 0.97),
        ('Legit - work/personal',  expand(B_WORK,150),  False, 0.97),
        ('Legit - marketing',      expand(B_MKT,120),   False, 0.95)]

GATE=True; rows=[]
for name,texts,expect,bar in CHECKS:
    ps=score(texts)
    ok=sum(1 for p in ps if (p>0.5)==expect); n=len(ps)
    rate=ok/n; lo,hi=wilson(ok,n)
    passed=rate>=bar; GATE=GATE and passed
    rows.append((name,rate,lo,hi,n,bar,passed))
    print(f"{'PASS' if passed else 'FAIL'}  {name:<24} {rate:6.1%}  "
          f"95% CI [{lo:.0%},{hi:.0%}]  n={n}  bar={bar:.0%}")

print()
print('GATE:','PASS' if GATE else 'FAIL')
if not GATE:
    print('Do not save. Options: raise BEC/security augmentation volume, add a')
    print('third epoch, or widen pool A structures. Note which bar failed --')
    print('a BEC failure and a security-notice failure need opposite fixes.')

## 7. Save only if the gate passed

In [ ]:
if GATE:
    torch.save(model.state_dict(),'roberta_phishing_model_v4.pth')
    import json as _json
    _json.dump([{'category':r[0],'rate':r[1],'ci_low':r[2],'ci_high':r[3],
                 'n':r[4],'bar':r[5],'passed':r[6]} for r in rows],
               open('v4_gate_results.json','w'),indent=2)
    from google.colab import files
    files.download('roberta_phishing_model_v4.pth')
    files.download('v4_gate_results.json')
    print('saved + downloading')
else:
    print('NOT saved — gate failed.')

## What to write in EVALUATION.md afterwards

Report the gate table **with the methodology caveat attached**, not bare:

> The v4 gate evaluates on template structures held out from training (pool B).
> Passing it demonstrates generalisation across BEC and security-notification
> *phrasing structures*, not validated detection of real-world BEC — real BEC
> corpora are scarce, which is why the augmentation is generated at all.
> Confirming real-world performance requires real samples and remains open.

Also record what v3 measured (subtle phishing 15.6% [9, 24]; security notices
90.0% [84, 94]) so the improvement is anchored to a measurement rather than
asserted, and keep the note that v3's original 7/7 and 12/12 gate was too small
to detect either problem — 0/12 has a 95% upper bound near 24%.